# Updating Beliefs: the Math of Bayes

Belief updating is Bayes' theorem — math you already know from Lesson 1, now wearing estimation clothes. Each piece gets a Bayesian name: **posterior** $P(A\mid B)$, **prior** $P(A)$, **likelihood** $P(B\mid A)$, and the **evidence** $P(B)$ computed by expanding over the cases. One coin flip updates "75% sure my coin is fair" into "65.2% sure," digit by digit — and the formula then generalizes to PMFs, PDFs, all four discrete/continuous combinations, and the parameter notation $\theta$ used across machine learning. It is based on the tenth video of Lesson 6 (Point Estimation).

## 1. Introduction

The three Bayesians' curves moved by magic; this notebook opens the machine. The engine is

$$
P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)}
$$

read the Bayesian way: $A$ is the event you want to predict (will you be offered the job?), $B$ is observable evidence that refines the prediction (were you asked to a follow-up call?).

After completing this notebook you will be able to:

- Name the four parts — posterior, likelihood, prior, evidence — and compute $P(B)$ by expanding over cases.
- Run the fair-vs-biased coin update by hand: prior 0.75 → posterior 0.652 after one head.
- Write the general PMF form, its PDF variants for the four discrete/continuous combinations, and the $\theta$ notation.
- Chain updates over many flips and watch beliefs converge.

## 2. Theory

### The four parts, with the job-offer story

$$
\underbrace{P(A \mid B)}_{\text{posterior}} = \frac{\overbrace{P(B \mid A)}^{\text{likelihood}}\;\overbrace{P(A)}^{\text{prior}}}{\underbrace{P(B)}_{\text{evidence}}}
$$

- **Posterior** $P(A\mid B)$: the refined probability of $A$ once $B$ is accounted for — your job chances *now that* you got the follow-up call.
- **Prior** $P(A)$: your belief before the evidence — job chances before the call request.
- **Likelihood** $P(B\mid A)$: how probable the evidence is if $A$ holds — chance of a follow-up call *given* you'll get the offer.
- **Evidence** $P(B)$: how probable the evidence is at all, usually computed by expanding:

$$
P(B) = P(B \mid A)P(A) + P(B \mid \neg A)P(\neg A)
$$

All the equation does: take prior beliefs and update them into posterior beliefs based on new evidence.

### The mystery coin, in full

Two coin types: **fair** ($P(\text{heads}) = 0.5$) and **biased** ($0.8$). Let $Y$ = the coin's heads-odds, taking values $0.5$ or $0.8$; let $X$ = the flip outcome (1 = heads, 0 = tails). Priors: most coins are fair — $P(Y{=}0.5) = 0.75$, $P(Y{=}0.8) = 0.25$.

Flip: **heads** ($x = 1$). Update the belief in fairness:

$$
P(Y{=}0.5 \mid X{=}1) = \frac{P(X{=}1 \mid Y{=}0.5)\,P(Y{=}0.5)}{P(X{=}1)}
= \frac{0.5 \times 0.75}{\underbrace{0.5 \times 0.75 + 0.8 \times 0.25}_{=\,0.575}} = 0.652
$$

You used to be 75% sure the coin was fair; one head nudges you down to **65.2%** — and the same computation gives $P(Y{=}0.8 \mid X{=}1) = 0.348$. The two posteriors sum to 1, as beliefs over an exhaustive pair must.

### Generalizing: PMFs, PDFs, and $\theta$

Rewriting each term as a probability mass function gives the general discrete-discrete form

$$
p_{Y \mid X}(y \mid x) = \frac{p_{X \mid Y}(x \mid y)\; p_Y(y)}{p_X(x)}
$$

valid for *any* event value $y$ and any evidence $x$. Four combinations then cover everything: replace each PMF with a PDF wherever the corresponding variable is **continuous** — discrete/discrete, continuous/continuous, and the two mixed cases. In every version the left side is the **posterior** and $p_Y$ (or $f_Y$) is the **prior**.

Finally, in machine-learning contexts the unknown is a model **parameter**, so $Y$ is conventionally renamed $\Theta$ (values $\theta$):

$$
f_{\Theta \mid X}(\theta \mid x) = \frac{p_{X \mid \Theta}(x \mid \theta)\, f_\Theta(\theta)}{p_X(x)}
$$

Notation only — the concepts are identical.

## 3. Worked Examples

### Example 1: the biased-side posterior

$$
P(Y{=}0.8 \mid X{=}1) = \frac{0.8 \times 0.25}{0.575} = \frac{0.20}{0.575} = 0.348
$$

Check: $0.652 + 0.348 = 1$. The denominator 0.575 is the same for both — it never changes the *ranking*, only the normalization (why MAP could ignore it).

### Example 2: a tails flip instead

$P(X{=}0) = 0.5(0.75) + 0.2(0.25) = 0.425$, and $P(Y{=}0.5 \mid X{=}0) = \frac{0.375}{0.425} = 0.882$ — tails is *evidence of fairness* (the biased coin rarely shows it), so belief in fairness rises from 0.75 to 0.88.

### Example 3: chaining updates

Yesterday's posterior is today's prior. After heads then tails: $0.75 \to 0.652 \to$ using prior 0.652: $\frac{0.5 \cdot 0.652}{0.5 \cdot 0.652 + 0.2 \cdot 0.348} = 0.824$. Two flips, two updates — and the order (HT vs TH) doesn't matter, as the exercises verify.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

coin_types = np.array([0.5, 0.8])       # heads probability of each type
prior = np.array([0.75, 0.25])          # belief: fair, biased

def update(prior, outcome):
    """One Bayes update for outcome 1 (heads) or 0 (tails)."""
    likelihood = np.where(outcome == 1, coin_types, 1 - coin_types)
    joint = likelihood * prior
    return joint / joint.sum()          # divide by the evidence P(x)

### The video's computation, digit by digit

In [ ]:
likelihood_heads = coin_types           # P(X=1 | Y)
evidence = (likelihood_heads * prior).sum()
posterior_fair = likelihood_heads[0] * prior[0] / evidence

print(f"P(X=1) = 0.5*0.75 + 0.8*0.25 = {evidence}")
print(f"P(Y=0.5 | X=1) = 0.5*0.75/{evidence} = {posterior_fair:.3f}")
post = update(prior, 1)
print(f"full posterior after one head: fair {post[0]:.3f}, biased {post[1]:.3f}"
      f"   (sum = {post.sum():.0f})")

### Chaining updates over a whole flip sequence

Posterior in, prior out: run the update once per flip of a genuinely biased coin and watch the belief in "fair" collapse.

In [ ]:
rng = np.random.default_rng(0)

true_p = 0.8
flips = (rng.random(50) < true_p).astype(int)

belief = prior.copy()
trajectory = [belief[0]]
for f in flips:
    belief = update(belief, f)
    trajectory.append(belief[0])

print(f"flips: {''.join('H' if f else 'T' for f in flips[:20])}...")
print(f"P(fair) after  1 flip : {trajectory[1]:.3f}")
print(f"P(fair) after 10 flips: {trajectory[10]:.3f}")
print(f"P(fair) after 50 flips: {trajectory[50]:.4f}")

plt.figure(figsize=(9, 4.2))
plt.plot(trajectory, "o-", markersize=3, color="tab:blue")
plt.axhline(0.75, color="gray", linestyle=":", label="prior 0.75")
plt.xlabel("number of flips observed")
plt.ylabel("P(coin is fair | data)")
plt.title("Chained Bayes updates: a biased coin talks its way out of the fair prior")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The continuous-$\theta$ version

Same machinery with a *density* prior over all $\theta \in [0,1]$ (the mixed case: continuous parameter, discrete evidence) — one update per flip reshapes the whole curve, reproducing the three-Bayesians pictures from the last notebook.

In [ ]:
theta = np.linspace(0.001, 0.999, 999)
prior_density = theta**9 * (1 - theta)**9            # the 'moderate' fair-ish prior
prior_density /= np.trapezoid(prior_density, theta)

def update_density(density, outcome):
    likelihood = theta if outcome == 1 else 1 - theta
    post = likelihood * density
    return post / np.trapezoid(post, theta)          # divide by evidence p(x)

density = prior_density.copy()
snapshots = {0: density.copy()}
for i, f in enumerate(flips, 1):
    density = update_density(density, f)
    if i in (1, 10, 50):
        snapshots[i] = density.copy()

plt.figure(figsize=(9.5, 4.6))
for (i, d), color in zip(snapshots.items(),
                         ["tab:gray", "tab:blue", "tab:orange", "tab:green"]):
    plt.plot(theta, d, color=color, linewidth=2,
             label=f"after {i} flips (MAP {theta[np.argmax(d)]:.2f})")
plt.axvline(0.8, color="black", linestyle=":", linewidth=1, label="true p = 0.8")
plt.xlabel(r"$\theta$ (heads probability)")
plt.ylabel("posterior density")
plt.title("Continuous prior, discrete evidence: the density migrates to the truth")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

## 5. Visualizations

### Anatomy of one update

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4), sharey=True)
labels = ["fair (0.5)", "biased (0.8)"]
colors = ["tab:blue", "tab:red"]

for ax, values, title in [
        (axes[0], prior, "prior  P(Y)"),
        (axes[1], coin_types * prior, "likelihood x prior\n(joint, sums to P(X=1) = 0.575)"),
        (axes[2], update(prior, 1), "posterior  P(Y | X=1)\n(joint / evidence)")]:
    ax.bar(labels, values, color=colors, edgecolor="black", alpha=0.85)
    for i, v in enumerate(values):
        ax.annotate(f"{v:.3f}", xy=(i, v), ha="center", va="bottom")
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3, axis="y")
plt.suptitle("One head: 0.75 -> 0.652. Multiply by the likelihood, divide by the evidence.")
plt.tight_layout()
plt.show()

### Heads pushes one way, tails the other

In [ ]:
post_h = update(prior, 1)
post_t = update(prior, 0)

plt.figure(figsize=(8, 4.2))
x = np.arange(3)
plt.bar(x - 0.15, [prior[0], post_h[0], post_t[0]], 0.3, color="tab:blue",
        label="P(fair)")
plt.bar(x + 0.15, [prior[1], post_h[1], post_t[1]], 0.3, color="tab:red",
        label="P(biased)")
plt.xticks(x, ["prior", "after HEADS", "after TAILS"])
plt.ylabel("belief")
plt.title("Evidence cuts both ways: heads favors bias (0.348), tails favors fairness (0.882... for fair)")
plt.grid(True, alpha=0.3, axis="y")
plt.legend()
plt.show()

### The whole story on one page

Everything above in a single sheet: the anatomy of one update, heads vs. tails, and a chain of 50 updates on the mystery coin.

<img src="img/09_bayesian_updating.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** Redo the video's computation for a **tails** first flip: find $P(X{=}0)$ and both posteriors, and check they sum to 1.

**Exercise 2.** Label each factor of $P(Y{=}0.8 \mid X{=}1) = \frac{0.8 \times 0.25}{0.575}$ with its Bayesian name, and state which of the four discrete/continuous formula variants this example uses.

### Intermediate

**Exercise 3.** Order doesn't matter: update on H then T, and on T then H, and confirm identical final beliefs. Then show why algebraically (what do the two chained numerators multiply to?).

**Exercise 4.** How many heads *in a row* until you believe the coin is more likely biased than fair ($P(\text{biased}) > 0.5$)? Solve by iteration and confirm with the odds form: posterior odds = prior odds × (0.8/0.5)^k.

### Advanced

**Exercise 5.** Three coin types: add a tails-biased coin ($P(\text{heads}) = 0.2$) with priors (0.7, 0.15, 0.15). Update on the sequence H,H,T,H and report the posterior after each flip. Which type leads at each stage?

**Exercise 6.** The evidence term as a predictor: before each flip, $P(\text{next flip is H}) = \sum_y P(H \mid y) P(y)$ under the *current* belief. Track this "predictive probability" along the 50-flip run and show it converges to 0.8 — Bayes not only learns the coin type, it learns to predict.

## 7. Solutions

In [ ]:
# Exercise 1: tails first
ev_t = (1 - coin_types) @ prior
post_t = (1 - coin_types) * prior / ev_t
print(f"P(X=0) = 0.5*0.75 + 0.2*0.25 = {ev_t}")
print(f"posteriors: fair {post_t[0]:.3f}, biased {post_t[1]:.3f}, "
      f"sum {post_t.sum():.0f}")

In [ ]:
# Exercise 2: naming the factors
print("Exercise 2: 0.8 = likelihood P(X=1|Y=0.8); 0.25 = prior P(Y=0.8);")
print("0.575 = evidence P(X=1); result = posterior P(Y=0.8|X=1).")
print("Both the parameter Y and the evidence X are discrete -> the PMF/PMF")
print("(discrete-discrete) variant of the four formulas.")

In [ ]:
# Exercise 3: commutativity of updates
ht = update(update(prior, 1), 0)
th = update(update(prior, 0), 1)
print(f"Exercise 3: H then T -> {ht.round(4)},  T then H -> {th.round(4)}")
print("Identical: chaining multiplies the prior by P(H|y) * P(T|y) -- a product,")
print("and products don't care about order. (Evidence denominators normalize at")
print("the end either way.)")

In [ ]:
# Exercise 4: heads in a row until biased leads
belief = prior.copy()
k = 0
while belief[1] <= 0.5:
    belief = update(belief, 1)
    k += 1
print(f"Exercise 4: biased overtakes after k = {k} straight heads "
      f"(P(biased) = {belief[1]:.3f})")
odds_k = np.log(3) / np.log(0.8 / 0.5)   # prior odds fair:biased = 3
print(f"odds form: (0.8/0.5)^k > 3  =>  k > {odds_k:.2f}  -> k = {int(np.ceil(odds_k))}")

In [ ]:
# Exercise 5: three coin types
types3 = np.array([0.5, 0.8, 0.2])
belief3 = np.array([0.70, 0.15, 0.15])
names3 = ["fair", "heads-biased", "tails-biased"]

for f, sym in [(1, "H"), (1, "H"), (0, "T"), (1, "H")]:
    lik = np.where(f == 1, types3, 1 - types3)
    belief3 = lik * belief3
    belief3 /= belief3.sum()
    lead = names3[int(np.argmax(belief3))]
    print(f"after {sym}: {np.round(belief3, 3)}   leader: {lead}")

In [ ]:
# Exercise 6: the predictive probability
belief = prior.copy()
predictive = [coin_types @ belief]
for f in flips:
    belief = update(belief, f)
    predictive.append(coin_types @ belief)

plt.figure(figsize=(9, 4))
plt.plot(predictive, color="tab:purple", linewidth=2)
plt.axhline(0.8, color="black", linestyle=":", label="true P(heads) = 0.8")
plt.xlabel("flips observed")
plt.ylabel("P(next flip = H | data so far)")
plt.title("Exercise 6: the evidence term doubles as a forecast -- and it learns")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"predictive after 0 flips: {predictive[0]:.3f} (0.575, the video's P(X=1))")
print(f"predictive after 50 flips: {predictive[-1]:.3f} -> 0.8")

## 8. Conclusion

- Belief updating is Bayes' theorem with named parts: **posterior** = **likelihood** × **prior** / **evidence**, with $P(B)$ expanded over the cases ($P(B|A)P(A) + P(B|\neg A)P(\neg A)$).
- The mystery coin, worked in full: priors $(0.75, 0.25)$, one head, evidence $0.575$ → posteriors $(0.652, 0.348)$, summing to 1. Tails would push the other way (fairness up to 0.882).
- Updates **chain** (posterior becomes prior; order irrelevant) and generalize: PMF↔PDF for each variable gives four formula variants, and renaming $Y$ to $\Theta$ gives the parameter-estimation notation of machine learning.
- The evidence denominator is constant across hypotheses (harmless for MAP) — and doubles as the *predictive* probability of the next observation, which itself converges to the truth.

Next: one more worked example of Bayesian updating in action.